# Capstone: Full-Stack App Builder [Production - Module 05]

> **MLCourse - Agentic AI - CrewAI Production**

This capstone project demonstrates a complete multi-agent system that takes
a software specification and generates a full-stack application. Six specialized
agents collaborate through a orchestrated pipeline: Product Manager, Architect,
Frontend Developer, Backend Developer, QA Engineer, and Tech Writer.

## What you will learn

1. Multi-agent orchestration with sequential and parallel execution.
2. Flow-based architecture with `@start`, `@listen`, and `@router` decorators.
3. Conditional routing: if QA finds critical bugs, route back to developers.
4. Human-in-the-loop approval gates after architecture design.
5. Persistent memory across crew runs.
6. Production-grade patterns: error handling, logging, guard cells.

## Architecture Overview

```
Product Manager --> Architect --> [Frontend Dev + Backend Dev] --> QA Engineer --> Tech Writer
                                             ^                        |
                                             |_______ if bugs ________|
```

All agents use Ollama (llama3.2) -- fully local, no API keys needed.
The generated "code" is a simple Todo API -- the focus is on orchestration.

In [ ]:
# ---- Setup: imports, environment, track discovery ---------------------------

import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
load_dotenv(TRACK / ".env", override=False)

api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] OPENAI_API_KEY found -- optional cloud calls will work")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

print("Track root:", TRACK)
print("Timestamp:", datetime.now().isoformat())

In [ ]:
# ---- Check Ollama availability --------------------------------------------

OLLAMA_OK = False
try:
    from langchain_ollama import ChatOllama
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    OLLAMA_OK = True
    print("Ollama: ONLINE (llama3.1:8b)")
except Exception as e:
    print("Ollama: OFFLINE --", e)
    print("Agent pipelines will run with stub outputs")

In [ ]:
# ---- CrewAI imports ---------------------------------------------------------

CREWAI_OK = False
try:
    from crewai import Agent, Task, Crew, Process, LLM
    from crewai import flow
    from crewai.flow.flow import Flow, listen, start, router
    CREWAI_OK = True
    print("CrewAI version:", __import__("crewai").__version__)
except ImportError as e:
    CREWAI_OK = False
    print("CrewAI not installed:", e)

## Phase 1: Sample Application Specification

We define a simple Todo API specification that the agents will process.
In production, this would come from a product team or client document.
The spec is intentionally simple so we can see the full pipeline output.

In [ ]:
# ---- Define the application specification ----------------------------------

APP_SPEC = {
    "name": "TodoAPI",
    "description": "A simple REST API for managing todo items",
    "version": "1.0.0",
    "requirements": [
        "Create, read, update, and delete todo items",
        "Mark items as complete or incomplete",
        "Filter items by status (all, active, completed)",
        "Each item has: id, title, description, completed, created_at",
        "JSON request and response format",
        "Input validation and error handling",
    ],
    "tech_stack": {
        "backend": "FastAPI (Python)",
        "frontend": "React (TypeScript)",
        "database": "SQLite (via SQLAlchemy)",
        "testing": "pytest + React Testing Library",
    },
    "constraints": [
        "Must be self-contained (no external APIs)",
        "Code must include type hints",
        "All endpoints must have error handling",
        "Frontend must be responsive",
    ],
}

print("=== Application Specification ===")
print(f"Name: {APP_SPEC['name']}")
print(f"Description: {APP_SPEC['description']}")
print(f"Version: {APP_SPEC['version']}")
print(f"Requirements: {len(APP_SPEC['requirements'])} items")
print(f"Tech stack: {list(APP_SPEC['tech_stack'].keys())}")
print(f"Constraints: {len(APP_SPEC['constraints'])} items")

## Phase 2: Agent Definitions

Each agent has a specific role, goal, and backstory. The LLM uses the
backstory to understand the agent's expertise and communication style.
All agents use Ollama (llama3.2) for local execution.

In [ ]:
# ---- Define all six agents -------------------------------------------------

if CREWAI_OK and OLLAMA_OK:
    ollama_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

    # Agent 1: Product Manager -- parses spec into user stories.
    pm_agent = Agent(
        role="Product Manager",
        goal=(
            "Parse the application specification into structured user stories "
            "with acceptance criteria. Ensure every requirement is covered."
        ),
        backstory=(
            "You are an experienced product manager who excels at translating "
            "business requirements into clear, actionable user stories. You "
            "always include acceptance criteria and edge cases."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    # Agent 2: Architect -- designs system architecture and DB schema.
    architect_agent = Agent(
        role="System Architect",
        goal=(
            "Design the system architecture, database schema, and API contracts "
            "based on the user stories. Output structured technical designs."
        ),
        backstory=(
            "You are a senior software architect with deep experience in "
            "REST API design, database modeling, and system integration. "
            "You produce clear, implementable technical specifications."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    # Agent 3: Frontend Developer -- generates React components.
    frontend_agent = Agent(
        role="Frontend Developer",
        goal=(
            "Generate React TypeScript components that implement the UI "
            "for the Todo application based on the API contract."
        ),
        backstory=(
            "You are a skilled React developer who writes clean TypeScript "
            "code. You use functional components, hooks, and proper typing. "
            "Your code is production-ready with error handling."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    # Agent 4: Backend Developer -- generates FastAPI endpoints.
    backend_agent = Agent(
        role="Backend Developer",
        goal=(
            "Generate FastAPI Python endpoints that implement the Todo API "
            "with SQLAlchemy models, input validation, and error handling."
        ),
        backstory=(
            "You are an expert Python developer specializing in FastAPI. "
            "You write clean, well-typed code with proper error handling, "
            "Pydantic models, and SQLAlchemy database integration."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    # Agent 5: QA Engineer -- writes tests and finds bugs.
    qa_agent = Agent(
        role="QA Engineer",
        goal=(
            "Write comprehensive tests for the generated code, run them, "
            "and report any bugs or issues found."
        ),
        backstory=(
            "You are a meticulous QA engineer who writes thorough test cases. "
            "You test happy paths, edge cases, and error conditions. You "
            "report bugs with clear reproduction steps."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    # Agent 6: Tech Writer -- creates documentation.
    writer_agent = Agent(
        role="Technical Writer",
        goal=(
            "Create comprehensive README documentation and API docs "
            "for the generated application."
        ),
        backstory=(
            "You are a technical writer who creates clear, concise documentation. "
            "You write README files with setup instructions, API reference "
            "documentation, and usage examples."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    print("All 6 agents defined:")
    agents = [pm_agent, architect_agent, frontend_agent, backend_agent, qa_agent, writer_agent]
    for a in agents:
        print(f"  - {a.role}")
else:
    print("[SKIP] CrewAI or Ollama not available")

## Phase 3: Task Definitions

Each task maps to an agent and has a description, expected output format,
and optional context from previous tasks. Tasks define the pipeline.

In [ ]:
# ---- Define all tasks ------------------------------------------------------

if CREWAI_OK and OLLAMA_OK:
    # Task 1: Product Manager creates user stories.
    pm_task = Task(
        description=(
            "Analyze the following application specification and create "
            "structured user stories with acceptance criteria.\n\n"
            "Specification:\n{spec}\n\n"
            "Output format: numbered list of user stories, each with:\n"
            "- Story title\n"
            "- As a [role], I want [feature], so that [benefit]\n"
            "- Acceptance criteria (bullet points)\n"
            "- Priority (high/medium/low)"
        ),
        expected_output=(
            "A structured list of user stories covering all requirements, "
            "each with title, user story format, acceptance criteria, and priority."
        ),
        agent=pm_agent,
    )

    # Task 2: Architect designs the system.
    architect_task = Task(
        description=(
            "Based on the user stories, design the system architecture.\n"
            "Include:\n"
            "1. Database schema (table names, columns, types, constraints)\n"
            "2. API endpoints (method, path, request/response schema)\n"
            "3. Project file structure\n"
            "4. Component diagram (text-based)\n\n"
            "Tech stack: FastAPI backend, React frontend, SQLite database."
        ),
        expected_output=(
            "Complete technical design with database schema, API contracts, "
            "file structure, and component diagram."
        ),
        agent=architect_agent,
        context=[pm_task],
    )

    # Task 3: Frontend Developer generates React code.
    frontend_task = Task(
        description=(
            "Generate React TypeScript components for the Todo application.\n"
            "Include:\n"
            "1. Main App component with state management\n"
            "2. TodoList component for displaying items\n"
            "3. TodoItem component for individual items\n"
            "4. AddTodo component for creating new items\n"
            "5. Filter component for status filtering\n"
            "6. API service module for backend communication\n\n"
            "Use functional components, hooks, and TypeScript types."
        ),
        expected_output=(
            "Complete React TypeScript code for all components, "
            "types, and API service module."
        ),
        agent=frontend_agent,
        context=[architect_task],
    )

    # Task 4: Backend Developer generates FastAPI code.
    backend_task = Task(
        description=(
            "Generate FastAPI Python code for the Todo API.\n"
            "Include:\n"
            "1. SQLAlchemy models for Todo items\n"
            "2. Pydantic schemas for request/response validation\n"
            "3. CRUD endpoints (GET, POST, PUT, DELETE)\n"
            "4. Filter endpoint for status-based filtering\n"
            "5. Error handling middleware\n"
            "6. Database initialization code\n\n"
            "Use type hints, docstrings, and proper error handling."
        ),
        expected_output=(
            "Complete FastAPI code with models, schemas, endpoints, "
            "error handling, and database setup."
        ),
        agent=backend_agent,
        context=[architect_task],
    )

    # Task 5: QA Engineer writes and runs tests.
    qa_task = Task(
        description=(
            "Write comprehensive tests for both frontend and backend.\n"
            "Include:\n"
            "1. Backend: pytest tests for all API endpoints\n"
            "2. Backend: test CRUD operations, validation, error handling\n"
            "3. Frontend: React component tests (description only)\n"
            "4. Integration test scenarios\n"
            "5. Bug report if any issues found\n\n"
            "Run the tests and report results."
        ),
        expected_output=(
            "Test files with comprehensive coverage, test results, "
            "and bug report (if issues found)."
        ),
        agent=qa_agent,
        context=[frontend_task, backend_task],
    )

    # Task 6: Tech Writer creates documentation.
    writer_task = Task(
        description=(
            "Create documentation for the Todo application.\n"
            "Include:\n"
            "1. README.md with project overview, setup instructions, usage\n"
            "2. API reference documentation for all endpoints\n"
            "3. Architecture overview (text-based diagram)\n"
            "4. Contributing guidelines\n\n"
            "Write clear, concise documentation suitable for developers."
        ),
        expected_output=(
            "Complete README.md and API documentation in markdown format."
        ),
        agent=writer_agent,
        context=[pm_task, architect_task, frontend_task, backend_task],
    )

    print("All 6 tasks defined:")
    tasks = [pm_task, architect_task, frontend_task, backend_task, qa_task, writer_task]
    for t in tasks:
        print(f"  - {t.agent.role}: {t.description[:60]}...")
else:
    print("[SKIP] CrewAI or Ollama not available")

## Phase 4: Sequential Crew (Baseline)

The simplest orchestration is sequential: PM -> Architect -> Frontend ->
Backend -> QA -> Writer. Each agent runs after the previous one finishes.
This is the baseline before we add parallelism and routing.

In [ ]:
# ---- Build and run the sequential crew ------------------------------------

if CREWAI_OK and OLLAMA_OK:
    sequential_crew = Crew(
        agents=[pm_agent, architect_agent, frontend_agent, backend_agent, qa_agent, writer_agent],
        tasks=[pm_task, architect_task, frontend_task, backend_task, qa_task, writer_task],
        process=Process.sequential,
        verbose=False,
    )

    spec_text = json.dumps(APP_SPEC, indent=2)
    print("Sequential crew assembled:")
    print(f"  Agents: {len(sequential_crew.agents)}")
    print(f"  Tasks: {len(sequential_crew.tasks)}")
    print(f"  Process: {sequential_crew.process}")
    print()
    print("Running sequential pipeline (this may take several minutes)...")
    print("=" * 60)

    start_time = time.time()
    try:
        result = sequential_crew.kickoff(inputs={"spec": spec_text})
        elapsed = time.time() - start_time
        print(f"\nSequential pipeline completed in {elapsed:.1f}s")
        print(f"Output length: {len(str(result))} chars")

        # Save the result.
        data_dir = TRACK / "03_agentic_ai" / "04_crewai" / "data"
        data_dir.mkdir(parents=True, exist_ok=True)
        result_file = data_dir / "capstone_sequential_result.txt"
        with open(result_file, "w", encoding="utf-8") as f:
            f.write(str(result))
        print(f"Result saved to: {result_file}")

        SEQ_RESULT = str(result)
        SEQUENTIAL_OK = True
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"\nSequential pipeline error after {elapsed:.1f}s: {e}")
        SEQ_RESULT = ""
        SEQUENTIAL_OK = False
else:
    print("[SKIP] CrewAI or Ollama not available")
    SEQ_RESULT = ""
    SEQUENTIAL_OK = False

## Phase 5: Flow-Based Orchestration with Parallel Execution

Flow-based orchestration uses decorators to define execution order:
- `@start`: marks the entry point of the flow.
- `@listen("method_name")`: runs after the named method completes.
- `@router("method_name")`: conditionally routes based on return value.

This enables parallel execution (two agents listening to the same trigger)
and conditional routing (QA can route back to devs if bugs are found).

In [ ]:
# ---- Define the Flow-based pipeline ---------------------------------------

if CREWAI_OK and OLLAMA_OK:
    class AppBuilderFlow(Flow):
        """Full-stack app builder with parallel dev and conditional routing.

        Flow diagram:
        PM -> Architect -> [Frontend + Backend] -> QA -> Writer
                                    ^              |
                                    |__ if bugs ___|
        """

        @start()
        def start_pipeline(self):
            """Entry point: run Product Manager to create user stories."""
            print("[Flow] Starting pipeline with Product Manager")
            spec_text = json.dumps(APP_SPEC, indent=2)

            pm_crew = Crew(
                agents=[pm_agent],
                tasks=[pm_task],
                process=Process.sequential,
                verbose=False,
            )
            result = pm_crew.kickoff(inputs={"spec": spec_text})
            self.pm_output = str(result)
            print(f"[Flow] PM output: {len(self.pm_output)} chars")
            return self.pm_output

        @listen(start_pipeline)
        def design_architecture(self, pm_output):
            """Run Architect after PM finishes."""
            print("[Flow] Running Architect after PM")
            architect_crew = Crew(
                agents=[architect_agent],
                tasks=[architect_task],
                process=Process.sequential,
                verbose=False,
            )
            result = architect_crew.kickoff(inputs={"spec": json.dumps(APP_SPEC, indent=2)})
            self.arch_output = str(result)
            print(f"[Flow] Architect output: {len(self.arch_output)} chars")
            return self.arch_output

        @listen(design_architecture)
        def develop_frontend(self, arch_output):
            """Run Frontend Developer after Architect finishes."""
            print("[Flow] Running Frontend Developer")
            fe_crew = Crew(
                agents=[frontend_agent],
                tasks=[frontend_task],
                process=Process.sequential,
                verbose=False,
            )
            result = fe_crew.kickoff(inputs={"spec": json.dumps(APP_SPEC, indent=2)})
            self.fe_output = str(result)
            print(f"[Flow] Frontend output: {len(self.fe_output)} chars")
            return self.fe_output

        @listen(design_architecture)
        def develop_backend(self, arch_output):
            """Run Backend Developer after Architect finishes (parallel with FE)."""
            print("[Flow] Running Backend Developer (parallel with Frontend)")
            be_crew = Crew(
                agents=[backend_agent],
                tasks=[backend_task],
                process=Process.sequential,
                verbose=False,
            )
            result = be_crew.kickoff(inputs={"spec": json.dumps(APP_SPEC, indent=2)})
            self.be_output = str(result)
            print(f"[Flow] Backend output: {len(self.be_output)} chars")
            return self.be_output

        @listen(develop_frontend, develop_backend)
        def run_qa(self, fe_output, be_output):
            """Run QA after both Frontend and Backend finish."""
            print("[Flow] Running QA Engineer (both devs finished)")
            qa_crew = Crew(
                agents=[qa_agent],
                tasks=[qa_task],
                process=Process.sequential,
                verbose=False,
            )
            result = qa_crew.kickoff(inputs={"spec": json.dumps(APP_SPEC, indent=2)})
            self.qa_output = str(result)
            has_critical_bugs = "critical" in self.qa_output.lower()
            print(f"[Flow] QA output: {len(self.qa_output)} chars")
            print(f"[Flow] Critical bugs found: {has_critical_bugs}")
            return self.qa_output

        @router(run_qa)
        def check_qa_results(self, qa_output):
            """Route based on QA results -- fix bugs or proceed to docs."""
            has_critical = "critical" in qa_output.lower()
            if has_critical:
                print("[Flow] Routing back to developers for bug fixes")
                return "fix_bugs"
            else:
                print("[Flow] QA passed -- proceeding to documentation")
                return "proceed"

        @listen("fix_bugs")
        def fix_bugs(self):
            """Re-run developers to fix critical bugs (simplified)."""
            print("[Flow] Re-running developers for bug fixes")
            # In production, this would pass the bug report to the devs.
            # Here we run a simplified fix cycle.
            self.fix_output = "Bug fixes applied based on QA report."
            return self.fix_output

        @listen(check_qa_results)
        def write_documentation(self, qa_output):
            """Write documentation after QA passes."""
            print("[Flow] Running Technical Writer")
            writer_crew = Crew(
                agents=[writer_agent],
                tasks=[writer_task],
                process=Process.sequential,
                verbose=False,
            )
            result = writer_crew.kickoff(inputs={"spec": json.dumps(APP_SPEC, indent=2)})
            self.docs_output = str(result)
            print(f"[Flow] Documentation output: {len(self.docs_output)} chars")
            return self.docs_output

    print("AppBuilderFlow class defined")
    print("Methods: start_pipeline -> design_architecture -> [develop_frontend + develop_backend] -> run_qa -> write_documentation")
else:
    print("[SKIP] CrewAI or Ollama not available")

## Phase 6: Human-in-the-Loop Approval Gate

After the Architect designs the system, we pause for human approval.
This is critical in production -- you want a human to review the
architecture before code generation begins.

In CrewAI Flows, HITL is implemented by returning a special value
that triggers a pause, or by using `input()` for notebook-based HITL.

In [ ]:
# ---- HITL approval gate pattern -------------------------------------------

if CREWAI_OK:
    class HITLAppBuilderFlow(Flow):
        """AppBuilderFlow with human-in-the-loop approval after architecture."""

        @start()
        def start_pipeline(self):
            print("[Flow] Starting pipeline")
            return "PM output placeholder"

        @listen(start_pipeline)
        def design_architecture(self, pm_output):
            print("[Flow] Architect designing system...")
            return "Architecture design complete"

        @listen(design_architecture)
        def human_approval_gate(self, arch_output):
            """HITL gate: pause for human review of architecture."""
            print("\n" + "=" * 60)
            print("HUMAN-IN-THE-LOOP APPROVAL GATE")
            print("=" * 60)
            print("Architecture design complete. Review the output.")
            print("In production, this would display the design for approval.")
            print()

            # In a notebook, we use input() for HITL.
            # In production, this would be an API call or webhook.
            try:
                # Guard against non-notebook environments.
                get_ipython
                in_notebook = True
            except NameError:
                in_notebook = False

            if in_notebook:
                # In notebook: ask for approval.
                # For demo, we auto-approve.
                print("[HITL] Auto-approving for demo (in production, wait for user)")
                approved = True
            else:
                # Not in notebook: auto-approve.
                print("[HITL] Not in notebook -- auto-approving")
                approved = True

            if approved:
                print("[HITL] Approved -- continuing to development")
                return arch_output
            else:
                print("[HITL] Rejected -- stopping pipeline")
                return None

        @listen(human_approval_gate)
        def develop_frontend(self, arch_output):
            if arch_output is None:
                print("[Flow] Pipeline stopped by human rejection")
                return None
            print("[Flow] Frontend development")
            return "Frontend code"

        @listen(human_approval_gate)
        def develop_backend(self, arch_output):
            if arch_output is None:
                return None
            print("[Flow] Backend development")
            return "Backend code"

    print("HITLAppBuilderFlow defined with approval gate after architecture")

## Phase 7: Persistent Memory

CrewAI supports persistent memory that survives across runs. This is
useful for agents that need to remember previous interactions, learn
from past mistakes, or maintain context across sessions.

Memory types:
- **Short-term memory**: within a single crew execution.
- **Long-term memory**: persists across crew runs (stored in files).
- **Entity memory**: remembers entities (people, places, concepts).

In [ ]:
# ---- Memory configuration -------------------------------------------------

if CREWAI_OK:
    print("=== CrewAI Memory System ===\n")
    print("Memory types:")
    print("  1. Short-term: within single execution (default)")
    print("  2. Long-term: persists across runs (file-based)")
    print("  3. Entity memory: remembers named entities")
    print()

    # Show how to enable memory on a crew.
    print("Enable memory on a crew:")
    print("  crew = Crew(")
    print("      agents=[...],")
    print("      tasks=[...],")
    print("      memory=True,")
    print("      LongTermMemory=SQLiteLongTermMemory(),")
    print("  )")
    print()

    # Show memory-enabled crew configuration.
    print("Memory-enabled crew for the capstone:")
    print("  - Long-term memory: stores past project decisions")
    print("  - Entity memory: remembers tech stack choices, API patterns")
    print("  - Short-term memory: maintains context within pipeline")

    # Demonstrate memory initialization pattern.
    memory_code = '''
from crewai import Crew, Process
from crewai.memory import LongTermMemory, ShortTermMemory, EntityMemory
from crewai.memory.storage import SQLiteStorage, RAGStorage

# Configure memory backends.
ltm_storage = SQLiteStorage(path="./memory/long_term.db")
stm_storage = SQLiteStorage(path="./memory/short_term.db")
entity_storage = RAGStorage(path="./memory/entities")

# Create crew with memory.
crew = Crew(
    agents=[...],
    tasks=[...],
    memory=True,
    long_term_memory=LongTermMemory(storage=ltm_storage),
    short_term_memory=ShortTermMemory(storage=stm_storage),
    entity_memory=EntityMemory(storage=entity_storage),
)
'''
    print("\nMemory initialization code:")
    print(memory_code)

## Phase 8: Error Handling and Guard Cells

Production notebooks must handle errors gracefully. Every cell that
depends on external services (Ollama, CrewAI) is wrapped in try/except.
This ensures the notebook can be read even if services are unavailable.

In [ ]:
# ---- Error handling patterns ----------------------------------------------

print("=== Production Error Handling Patterns ===\n")

# Pattern 1: Guard cell for LLM availability.
print("Pattern 1: LLM guard cell")
print("  try:")
print("      llm = ChatOllama(model='llama3.1:8b', temperature=0)")
print("      llm.invoke('ping')")
print("      LLM_OK = True")
print("  except Exception as e:")
print("      LLM_OK = False")
print("      print(f'LLM unavailable: {e}')")
print()

# Pattern 2: Crew execution with error handling.
print("Pattern 2: Crew execution guard")
print("  try:")
print("      result = crew.kickoff(inputs={...})")
print("  except Exception as e:")
print("      print(f'Crew execution failed: {e}')")
print("      result = None")
print()

# Pattern 3: Tool execution with timeout.
print("Pattern 3: Tool timeout guard")
print("  try:")
print("      result = tool.execute(code=code, timeout=30)")
print("  except TimeoutError:")
print("      print('Tool execution timed out')")
print("  except Exception as e:")
print("      print(f'Tool error: {e}')")

## Phase 9: Running the Complete Pipeline

We run the full pipeline and collect all outputs. The pipeline generates
user stories, architecture design, frontend code, backend code, tests,
and documentation. All outputs are saved for review.

In [ ]:
# ---- Run the complete pipeline (sequential fallback) ----------------------

if CREWAI_OK and OLLAMA_OK:
    print("=== Running Full-Stack App Builder Pipeline ===\n")
    print("Pipeline stages:")
    print("  1. Product Manager -> User Stories")
    print("  2. Architect -> System Design")
    print("  3. Frontend Developer -> React Components")
    print("  4. Backend Developer -> FastAPI Endpoints")
    print("  5. QA Engineer -> Tests + Bug Report")
    print("  6. Tech Writer -> Documentation")
    print()

    spec_text = json.dumps(APP_SPEC, indent=2)

    # For the demo, we run a simplified pipeline with just PM and Architect
    # to keep execution time reasonable.
    demo_crew = Crew(
        agents=[pm_agent, architect_agent],
        tasks=[pm_task, architect_task],
        process=Process.sequential,
        verbose=False,
    )

    print("Running simplified demo (PM + Architect only)...")
    print("Full pipeline would run all 6 agents sequentially.")
    print("=" * 60)

    start_time = time.time()
    try:
        demo_result = demo_crew.kickoff(inputs={"spec": spec_text})
        elapsed = time.time() - start_time
        print(f"\nDemo pipeline completed in {elapsed:.1f}s")
        print(f"Output length: {len(str(demo_result))} chars")
        print()
        print("=== Pipeline Output Preview ===")
        output_preview = str(demo_result)[:1000]
        print(output_preview)
        if len(str(demo_result)) > 1000:
            print("... (truncated)")

        # Save the full output.
        data_dir = TRACK / "03_agentic_ai" / "04_crewai" / "data"
        data_dir.mkdir(parents=True, exist_ok=True)
        output_file = data_dir / "capstone_pipeline_output.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(str(demo_result))
        print(f"\nFull output saved to: {output_file}")

    except Exception as e:
        elapsed = time.time() - start_time
        print(f"\nPipeline error after {elapsed:.1f}s: {e}")
        print("This is expected if Ollama is running slow or out of memory.")
else:
    print("[SKIP] CrewAI or Ollama not available")

## Phase 10: Output Analysis and Metrics

After the pipeline runs, we analyze the outputs for quality metrics:
completeness, code structure, documentation coverage, and test coverage.

In [ ]:
# ---- Output analysis patterns ---------------------------------------------

if CREWAI_OK:
    print("=== Output Quality Metrics ===\n")

    def analyze_output(text, label):
        """Compute basic quality metrics for agent output."""
        lines = text.strip().split("\n")
        words = text.split()
        code_blocks = text.count("```")
        headers = sum(1 for l in lines if l.startswith("#"))
        bullet_points = sum(1 for l in lines if l.strip().startswith("-") or l.strip().startswith("*"))

        metrics = {
            "label": label,
            "lines": len(lines),
            "words": len(words),
            "code_blocks": code_blocks // 2,  # pairs of opening/closing
            "headers": headers,
            "bullet_points": bullet_points,
            "has_types": "def " in text and ":" in text,
            "has_docstrings": '"""' in text or "'''" in text,
        }
        return metrics

    # Analyze a sample output.
    sample_output = """
# User Stories

## Story 1: Create Todo
- As a user, I want to create a new todo item
- Acceptance criteria:
  - POST /todos accepts title and description
  - Returns 201 with created item
  - Returns 400 for invalid input

## Story 2: List Todos
- As a user, I want to see all my todos
- Acceptance criteria:
  - GET /todos returns all items
  - Supports ?status=active filter
"""

    metrics = analyze_output(sample_output, "User Stories")
    print("Sample output metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

    print()
    print("Quality checklist for generated code:")
    checklist = [
        ("Type hints present", "Has def and : annotations"),
        ("Docstrings present", "Has triple-quote strings"),
        ("Error handling", "Has try/except blocks"),
        ("Input validation", "Has Pydantic models"),
        ("Tests included", "Has assert or test_ functions"),
        ("README included", "Has # headers and setup instructions"),
    ]
    for item, check in checklist:
        print(f"  [ ] {item:30s} -- {check}")

## Phase 11: Production Deployment Checklist

Before deploying the generated application:

- [ ] All generated code has been reviewed by a human.
- [ ] Tests pass locally (`pytest` for backend, `npm test` for frontend).
- [ ] No hardcoded secrets or API keys in generated code.
- [ ] Error handling covers all edge cases.
- [ ] Documentation is complete and accurate.
- [ ] Database migrations are reversible.
- [ ] Logging is configured for production.
- [ ] Rate limiting is in place for API endpoints.

In [ ]:
# ---- Deployment checklist -------------------------------------------------

print("=== Production Deployment Checklist ===\n")
deploy_checklist = [
    "Human code review completed",
    "Backend tests pass (pytest)",
    "Frontend tests pass (npm test)",
    "No hardcoded secrets",
    "Error handling covers edge cases",
    "Documentation complete",
    "Database migrations reversible",
    "Logging configured",
    "Rate limiting enabled",
    "Security review completed",
]
for idx, item in enumerate(deploy_checklist, 1):
    print(f"  {idx:2d}. [ ] {item}")

## Phase 12: Summary and Architecture Recap

This capstone demonstrated a complete multi-agent full-stack app builder:

1. **Product Manager Agent**: parsed spec into user stories with acceptance criteria.
2. **Architect Agent**: designed system architecture, DB schema, API contracts.
3. **Frontend Developer Agent**: generated React TypeScript components.
4. **Backend Developer Agent**: generated FastAPI Python endpoints.
5. **QA Engineer Agent**: wrote tests, ran them, reported bugs.
6. **Tech Writer Agent**: created README and API documentation.

Orchestration patterns demonstrated:
- Sequential execution (baseline).
- Flow-based orchestration with `@start`, `@listen`, `@router`.
- Parallel execution (Frontend + Backend running simultaneously).
- Conditional routing (QA routes back to devs if critical bugs found).
- Human-in-the-loop approval gate after architecture design.
- Persistent memory across runs.
- Error handling and guard cells throughout.

In [ ]:
# ---- Final summary --------------------------------------------------------

print("=== Capstone Summary ===\n")
print("Full-Stack App Builder Pipeline:")
print("  Input: Application specification (JSON)")
print("  Output: User stories, architecture, code, tests, docs")
print()
print("Agents: 6 (PM, Architect, Frontend, Backend, QA, Writer)")
print("Pattern: Flow-based with parallel execution")
print("LLM: Ollama llama3.1:8b (local, free)")
print("HITL: Approval gate after architecture")
print("Memory: Persistent across runs")
print()
print("Key files generated:")
data_dir = TRACK / "03_agentic_ai" / "04_crewai" / "data"
files = [
    "crew_training_data.jsonl",
    "crew_trace.json",
    "AGENTS.md",
    "capstone_sequential_result.txt",
    "capstone_pipeline_output.txt",
]
for f in files:
    path = data_dir / f
    exists = path.exists()
    print(f"  {'[x]' if exists else '[ ]'} {f}")

print()
print("Next steps:")
print("  - Add real code generation (not demo stubs)")
print("  - Implement retry logic for failed agents")
print("  - Add streaming output for real-time progress")
print("  - Build a web UI for specification input")
print("  - Deploy generated apps to cloud hosting")